In [ ]:
print("start")
!pip install --user transformers datasets torch scikit-learn pandas numpy matplotlib seaborn
print("end")

In [ ]:
# ── Exploratory Data Analysis (EDA) ──────────────────────────────────────────

# Check class distribution
print("Label Distribution:")
print(df['label'].value_counts())
print(f"\nSpam %: {df['label'].value_counts(normalize=True)['spam']*100:.2f}%")

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Distribution (Ham vs Spam)', fontsize=13)
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Message length distribution
df['msg_length'] = df['message'].apply(len)
df.groupby('label')['msg_length'].plot(kind='hist', bins=40, alpha=0.6, ax=axes[1], legend=True)
axes[1].set_title('Message Length Distribution', fontsize=13)
axes[1].set_xlabel('Character Length')

plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nSample messages:")
print(df[df['label']=='spam']['message'].values[0])
print(df[df['label']=='ham']['message'].values[0])

In [ ]:
# ── Encode labels: ham → 0, spam → 1 ─────────────────────────────────────────
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})

# ── Extract features and labels ───────────────────────────────────────────────
texts = df['message'].tolist()
labels = df['label_enc'].tolist()

# ── Train / Validation / Test Split ──────────────────────────────────────────
# 70% train | 15% validation | 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.30, random_state=SEED, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Training samples  : {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples      : {len(X_test)}")

In [ ]:
# ── Load pre-trained BERT tokenizer ──────────────────────────────────────────
MODEL_NAME = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_LEN    = 128   # Max token length (BERT supports up to 512)
BATCH_SIZE = 16    # Batch size for training
EPOCHS     = 3     # Number of training epochs
LR         = 2e-5  # Learning rate (standard for BERT fine-tuning)

# ── Test tokenizer on a sample sentence ──────────────────────────────────────
sample = "Congratulations! You've won a FREE iPhone. Click now!"
tokens = tokenizer.encode_plus(
    sample,
    max_length=MAX_LEN,
    truncation=True,
    padding='max_length',
    return_tensors='pt'
)
print("Input IDs shape   :", tokens['input_ids'].shape)
print("Attention mask    :", tokens['attention_mask'].shape)
print("Sample tokens     :", tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])[:15])

In [ ]:
class SMSDataset(Dataset):
    """
    Custom PyTorch Dataset for SMS spam classification.
    Tokenizes each message using BERT tokenizer.
    """
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        # Return total number of samples
        return len(self.texts)

    def __getitem__(self, idx):
        text  = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenize the text with padding and truncation
        encoding = self.tokenizer.encode_plus(
            text,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            add_special_tokens=True,   # Adds [CLS] and [SEP] tokens
            return_attention_mask=True,
            return_tensors='pt'        # Return PyTorch tensors
        )

        return {
            'input_ids'      : encoding['input_ids'].squeeze(0),
            'attention_mask' : encoding['attention_mask'].squeeze(0),
            'label'          : torch.tensor(label, dtype=torch.long)
        }


# ── Create Dataset objects ────────────────────────────────────────────────────
train_dataset = SMSDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = SMSDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = SMSDataset(X_test,  y_test,  tokenizer, MAX_LEN)

# ── Create DataLoaders ────────────────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches     : {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches      : {len(test_loader)}")

In [ ]:
# ── Load BERT with a classification head (2 output classes) ──────────────────
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,           # Binary classification: ham or spam
    output_attentions=False,
    output_hidden_states=False
)

# Move model to GPU (if available)
model = model.to(device)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ── AdamW optimizer (HuggingFace recommended for BERT) ───────────────────────
# Weight decay is applied to prevent overfitting
optimizer = AdamW(
    model.parameters(),
    lr=LR,
    eps=1e-8,
    weight_decay=0.01
)

# ── Total training steps ─────────────────────────────────────────────────────
total_steps = len(train_loader) * EPOCHS

# ── Learning rate scheduler with linear warmup ───────────────────────────────
# Warmup helps BERT adapt to the task before aggressive updates
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),  # 10% warmup
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps        : {int(0.1 * total_steps)}")

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    """
    Runs one epoch of training.
    Returns: average loss and accuracy for the epoch.
    """
    model.train()  # Set model to training mode
    total_loss, correct, total = 0, 0, 0

    for batch in loader:
        # Move batch tensors to device
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch   = batch['label'].to(device)

        # Zero gradients before forward pass
        optimizer.zero_grad()

        # Forward pass through BERT
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )

        loss   = outputs.loss
        logits = outputs.logits

        # Backward pass and gradient update
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        scheduler.step()

        # Track metrics
        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == labels_batch).sum().item()
        total      += labels_batch.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, loader, device):
    """
    Evaluates the model on validation/test set.
    Returns: average loss, accuracy, all predictions, all true labels.
    """
    model.eval()  # Set model to evaluation mode
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():  # No gradient computation needed
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_batch   = batch['label'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels_batch
            )

            loss   = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds       = torch.argmax(logits, dim=1)
            correct    += (preds == labels_batch).sum().item()
            total      += labels_batch.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy, all_preds, all_labels

In [ ]:
# ── Main Training Loop ────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

print("🚀 Starting BERT Fine-Tuning...\n")
print(f"{'Epoch':<6} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<10}")
print("-" * 55)

for epoch in range(1, EPOCHS + 1):
    # Training phase
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)

    # Validation phase
    val_loss, val_acc, _, _ = evaluate(model, val_loader, device)

    # Save history for plotting
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"{epoch:<6} {train_loss:<12.4f} {train_acc:<12.4f} {val_loss:<12.4f} {val_acc:<10.4f}")

    # Save the best model based on validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_bert_model.pt')
        print(f"       ✅ Best model saved (val_acc={val_acc:.4f})")

print(f"\n🏆 Best Validation Accuracy: {best_val_acc:.4f}")

In [ ]:
# ── Load the best saved model ─────────────────────────────────────────────────
model.load_state_dict(torch.load('best_bert_model.pt', map_location=device))

# ── Evaluate on test set ──────────────────────────────────────────────────────
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, device)

print(f"📊 Test Loss    : {test_loss:.4f}")
print(f"📊 Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"📊 F1 Score     : {f1_score(test_labels, test_preds, average='weighted'):.4f}")

# ── Detailed Classification Report ───────────────────────────────────────────
print("\n📋 Classification Report:")
print(classification_report(test_labels, test_preds, target_names=['ham', 'spam']))

In [ ]:
# ── Confusion Matrix Heatmap ──────────────────────────────────────────────────
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['ham', 'spam'],
    yticklabels=['ham', 'spam']
)
plt.title('Confusion Matrix – BERT Spam Classifier', fontsize=13)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
def predict_message(text, model, tokenizer, max_len, device):
    """
    Predicts whether a given SMS message is spam or ham.
    Returns the predicted label and confidence score.
    """
    model.eval()

    # Tokenize the input text
    encoding = tokenizer.encode_plus(
        text,
        max_length=max_len,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )

    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
        pred    = int(np.argmax(probs))

    label = 'SPAM 🚨' if pred == 1 else 'HAM ✅'
    confidence = probs[pred] * 100
    return label, confidence


# ── Test on custom messages ───────────────────────────────────────────────────
test_messages = [
    "WINNER!! You have been selected for a FREE prize. Call now!",
    "Hey, are you coming to the meeting at 3pm?",
    "Urgent! Your account has been compromised. Click to verify!",
    "Can you pick up some groceries on your way home?",
    "Congratulations! You've won a $1000 Walmart gift card. Claim now."
]

print("🔍 Inference Results:\n")
print(f"{'Message':<55} {'Prediction':<12} {'Confidence':<10}")
print("-" * 80)

for msg in test_messages:
    label, conf = predict_message(msg, model, tokenizer, MAX_LEN, device)
    print(f"{msg[:52]:<55} {label:<12} {conf:.1f}%")

In [ ]:
# ── Save fine-tuned model and tokenizer for future use ────────────────────────
save_dir = './bert_spam_classifier'
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)      # Saves model weights + config
tokenizer.save_pretrained(save_dir)  # Saves tokenizer vocab + config

print(f"✅ Model and tokenizer saved to: {save_dir}/")
print("Files saved:")
for f in os.listdir(save_dir):
    print(f"  - {f}")